## Config — Lab 12
Shared configuration for all Lab 12 notebooks. It does not process data.
Catalog and schemas come from job parameters (bundle variables of the target).
Everything that is the same in dev and prod comes from `config/rag_config.yml`.

In [0]:
import logging

logger = logging.getLogger("lab12_rag")

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s - %(message)s"))
    logger.addHandler(handler)

logger.setLevel(logging.INFO)
logger.propagate = False


In [0]:
for key in ["catalog", "silver_schema", "gold_schema", "rag_schema"]:
    dbutils.widgets.text(key, "")

In [0]:
import os
import yaml
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

config_path = os.path.abspath(os.path.join(os.getcwd(), "..", "config", "rag_config.yml"))

with open (config_path, "r") as f:
    config = yaml.safe_load(f)

In [0]:
for key in ["catalog", "silver_schema", "gold_schema", "rag_schema"]:
    value = dbutils.widgets.get(key)
    if not value:
        raise ValueError(f"Parameter '{key}' is empty: pass it as a job parameter or fill the widget")
    config[key] = value

In [0]:
catalog = config["catalog"]
silver_schema = config["silver_schema"]
gold_schema = config["gold_schema"]
rag_schema = config["rag_schema"]

silver_table = f"{catalog}.{silver_schema}.{config['source']['silver_table']}"
category_dim_table = f"{catalog}.{gold_schema}.{config['source']['category_dim']}"

documents_table = f"{catalog}.{rag_schema}.{config['tables']['documents']}"
chunks_staging_table = f"{catalog}.{rag_schema}.{config['tables']['chunks_staging']}"
chunks_table = f"{catalog}.{rag_schema}.{config['tables']['chunks']}"
qa_logs_table = f"{catalog}.{rag_schema}.{config['tables']['qa_logs']}"

index_name = f"{catalog}.{rag_schema}.{config['ai_search']['index_name']}"

logger.info("Config loaded: source=%s, chunks=%s, index=%s", silver_table, chunks_table, index_name)
